# 04.1.4 - Supervised Learning: Klasifikasi Akurasi Tertinggi dengan ANN

## 1. Pengertian Artificial Neural Network (ANN)
**Artificial Neural Network (ANN)** atau Jaringan Saraf Tiruan adalah algoritma *Machine Learning* yang terinspirasi dari cara kerja otak manusia. Di *Scikit-Learn*, model ini sering disebut sebagai **MLP (*Multi-Layer Perceptron*)**.

**Analogi Sederhana:**
Bayangkan sebuah pabrik dengan beberapa lapisan pegawai:
1. **Lapisan Input:** Pegawai di pintu depan yang menerima data pelanggan (umur, total belanja, dsb).
2. **Lapisan Tersembunyi (*Hidden Layer*):** Pegawai spesialis di ruang tengah. Lapisan ini saling berdiskusi; ada yang khusus mencari pola kapan pelanggan belanja, ada yang khusus menghitung rata-rata pengeluaran. Mereka mengolah informasi secara berulang-ulang.
3. **Lapisan Output:** Manajer di pintu belakang yang menerima laporan dari ruang tengah dan memutuskan, *"Oh, pelanggan ini masuk ke Cluster 3!"*

## 2. Mengapa Menggunakan ANN?
Dari semua model yang kita uji (*Decision Tree*, *SVM*, *AdaBoost*), ANN adalah algoritma yang paling "kompleks" (*Black-Box*). Kita menggunakannya murni untuk mencari **akurasi tebakan tertinggi**. Model ini sangat pintar dalam memetakan pola hubungan antar data yang sangat rumit yang tidak bisa dilihat oleh manusia atau algoritma sederhana lainnya.

In [2]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. BACA DATA (Gunakan Pemenang Clustering: Standard K-Means)
df = pd.read_csv('../data/Labeled/hasildata_kmeans-standard.csv')

# Pisahkan Fitur (Var1 - Var11) dan Target (Cluster)
fitur = [f'Var{i}' for i in range(1, 12)]
X = df[fitur]
y = df['Cluster']

# Splitting Data: 80% Train, 20% Test (Menggunakan seluruh baris data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Proses Scaling (Wajib untuk ANN)
Sama persis seperti SVM, algoritma ANN sangat sensitif terhadap perbedaan besaran angka (misalnya angka jutaan pada 'Total Belanja' disandingkan dengan angka satuan pada 'Jumlah Kunjungan'). 

Jika kita tidak menyetarakan angkanya (*scaling*), "pegawai" di dalam ANN akan kebingungan dan mengira angka yang besar adalah data yang paling penting. Oleh karena itu, kita wajib menggunakan `StandardScaler`.

In [3]:
# 2. SCALING DATA
scaler_ann = StandardScaler()

# Model scaler belajar dari data latih (Train) sekaligus menyetarakan angkanya
X_train_scaled = scaler_ann.fit_transform(X_train)

# Data ujian (Test) juga disetarakan agar formatnya sama dengan data latih
X_test_scaled = scaler_ann.transform(X_test)

## 4. Training Model ANN
Sekarang kita mulai melatih modelnya. Ada dua pengaturan penting yang kita buat di sini:
* `hidden_layer_sizes=(100,)`: Kita menyewa 100 "pegawai spesialis" di ruang tengah untuk mencari pola pelanggan.
* `max_iter=500`: Kita memberi model kesempatan belajar (mengulang materi) maksimal sebanyak 500 kali putaran. 

*Catatan:* Jika nanti muncul teks kotak merah/pink berupa `ConvergenceWarning`, abaikan saja. Itu bukan *error*, melainkan sekadar info bahwa mesin merasa masih bisa belajar lebih lama lagi melebihi 500 putaran, namun karena kita batasi, dia berhenti. Hasilnya pun sudah sangat optimal.

In [4]:
# 3. TRAINING MODEL ANN
# Menggunakan max_iter=500 agar model punya cukup waktu belajar
model_ann = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
model_ann.fit(X_train_scaled, y_train)

# Evaluasi model menggunakan data ujian
prediksi_ann = model_ann.predict(X_test_scaled)
print("=== CLASSIFICATION REPORT: ANN (STANDARD K-MEANS) ===\n")
print(classification_report(y_test, prediksi_ann))

=== CLASSIFICATION REPORT: ANN (STANDARD K-MEANS) ===

              precision    recall  f1-score   support

           0       0.98      0.97      0.97       137
           1       0.97      0.98      0.97       211
           2       0.98      0.97      0.97       121
           3       0.99      0.99      0.99       166
           4       0.99      0.98      0.98       149
           5       0.97      1.00      0.98        83

    accuracy                           0.98       867
   macro avg       0.98      0.98      0.98       867
weighted avg       0.98      0.98      0.98       867



c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


### Analisis Hasil: Jawara Akurasi!
Luar biasa! Rapor performa di atas menunjukkan akurasi mencapai **98%** (atau `0.98`). 

Ini adalah algoritma dengan tebakan paling jitu dibandingkan tiga model sebelumnya. ANN berhasil memahami pola kebiasaan pelanggan dengan nyaris sempurna. Model ini adalah kandidat terkuat untuk diintegrasikan ke dalam Web UI sebagai otak utama sistem klasifikasi.

## 5. Ekspor Model dan Scaler
Karena ANN juga membutuhkan *Scaler*, kita wajib menyimpan **dua file** ini agar nantinya Web UI bisa memproses data pelanggan baru dengan lancar:
1. `scaler_ann_standard.pkl` (Untuk meratakan angka input dari pengguna UI).
2. `model_ann_classification_standard.pkl` (Otak penebaknya).

In [5]:
# 4. EXPORT SCALER & MODEL KE FOLDER 'models'
os.makedirs('../models', exist_ok=True)

# Simpan Scaler
joblib.dump(scaler_ann, '../models/scaler_ann_standard.pkl')

# Simpan Model ANN
joblib.dump(model_ann, '../models/model_ann_classification_standard.pkl')

print("\n[SUCCESS] Scaler & Model ANN berhasil diekspor ke folder '../models/'!")


[SUCCESS] Scaler & Model ANN berhasil diekspor ke folder '../models/'!
